## Results

In [1]:
%run ../utils/sampling.py
import pandas as pd

product_info_df = pd.read_csv('../data/cleaned/product-features.csv')
departments = pd.read_csv('../dataset/departments.csv')
products = pd.read_csv('../dataset/products.csv')

full_products_df = product_info_df.merge(products, on='product_id', how='left')

product_names_to_search = [
    "Coke",
    "Bananas",
    "Organic Whole Milk",
    "Plain Bagels",
    "Spaghetti"
]

top_ids = get_top_product_ids(full_products_df, product_names_to_search)
print(top_ids)

found_products = search_products(full_products_df, "Coke")
print(found_products.head(1))

found_products = search_products(full_products_df, "Eggo")
print(found_products.head(1))

{'Coke': {'query': 'Coke', 'matched_name': 'Diet Coke', 'product_id': 43631, 'order_penetration_pct': 0.2090284098225933}, 'Bananas': {'query': 'Bananas', 'matched_name': 'Banana', 'product_id': 24852, 'order_penetration_pct': 14.69933191782944}, 'Organic Whole Milk': {'query': 'Organic Whole Milk', 'matched_name': 'Organic Whole Milk', 'product_id': 27845, 'order_penetration_pct': 4.289592686991776}, 'Plain Bagels': {'query': 'Plain Bagels', 'matched_name': 'Plain Bagels', 'product_id': 20738, 'order_penetration_pct': 0.3195770658507922}, 'Spaghetti': {'query': 'Spaghetti', 'matched_name': 'Spaghetti', 'product_id': 32734, 'order_penetration_pct': 0.49186997686379}}
   product_name  product_id  order_penetration_pct
0  Coke Classic       16696               0.335068
             product_name  product_id  order_penetration_pct
0  Eggo Homestyle Waffles       30696               0.276216


In [2]:
product_ids = [
    info["product_id"] 
    for info in top_ids.values() 
    if info is not None
]

product_ids.append(16696)
product_ids.append(30696)
print(product_ids)
sampled_products_df = pd.DataFrame(product_ids, columns=['product_id'])
sampled_products_df.to_csv("../results/sampled-products.csv", index=None)

[43631, 24852, 27845, 20738, 32734, 16696, 30696]


In [3]:
%run ../utils/pairwise.py

# Get product_ids as a series
products = product_ids.copy()
product_df = pd.read_csv('../data/cleaned/product-info-full.csv')
orders_full_df = pd.read_csv('../dataset/order_products__prior.csv')

order_product_df = orders_full_df[['order_id', 'product_id']]

# Compute pairwise probabilties for focus products
compute_pairwise_probabilities_sample(order_product_df, 
    products,
    output_csv="../results/products-pairwise.csv",
    batch_size=1000
)

100%|██████████| 1/1 [00:04<00:00,  4.20s/it]

Completed computation. Saved to ../results/products-pairwise.csv


In [ ]:
%run ../utils/substitutes.py

dept1_df = pd.read_csv('../data/cleaned/pairwise-dept1.csv')
dept3_df = pd.read_csv('../data/cleaned/pairwise-dept3.csv')
dept4_df = pd.read_csv('../data/cleaned/pairwise-dept4.csv')
dept7_df = pd.read_csv('../data/cleaned/pairwise-dept7.csv')
dept9_df = pd.read_csv('../data/cleaned/pairwise-dept9.csv')
dept16_df = pd.read_csv('../data/cleaned/pairwise-dept16.csv')

pairwise_df = pd.concat([dept1_df, dept3_df, dept4_df, dept7_df, dept9_df, dept16_df], ignore_index=True)

similarity_df = pd.read_csv('../data/cleaned/product-similiarity.csv')

compute_sub_score(
    product_df,
    pairwise_df,
    products,
    similarity_df,
    "../results/raw-substitutes.csv")

Completed substitute calculations. Saved to ../results/raw-substitutes.csv


In [82]:
%run ../utils/substitutes.py

substitutes_df = pd.read_csv("../results/raw-substitutes.csv")

# Find the best substitution score threshold which will identify a product as a true substitute
best_threshold = find_best_threshold(substitutes_df)

# Add a new column 'identified_substitute' based on the threshold
substitutes_df['identified_substitute'] = substitutes_df['score'] >= best_threshold

# Compute transferability %
results = compute_transferability(substitutes_df, top_n=5)
results.to_csv("../results/substitutes-transfer.csv", index=False)

 Best threshold determined as: 0.398 and adjusted to: 0.41074705927046157



In [ ]:
%run ../utils/results.py

# Print the results
show_sub_results(results)


Product: Coke Classic  (ID: 16696)


,sub_name,transferability_pct,score,aisle
0,Classic Soda,0.149826,0.701174,soft drinks
1,Coke Zero,0.144124,0.674490,soft drinks
2,Coke,0.141851,0.663852,soft drinks
3,Cherry Coke,0.134633,0.630075,soft drinks
4,Vanilla Coke Zero,0.130740,0.611854,soft drinks



Product: Plain Bagels  (ID: 20738)


,sub_name,transferability_pct,score,aisle
0,Plain Mini Bagels,0.154851,0.737898,breakfast bakery
1,Plain Pre-Sliced Bagels,0.154590,0.736655,breakfast bakery
2,Assorted Bagels,0.145627,0.693944,breakfast bakery
3,Bagels Plain Presliced,0.143973,0.686062,breakfast bakery
4,Onion Bagels,0.138857,0.661686,breakfast bakery



Product: Banana  (ID: 24852)


...

,sub_name,transferability_pct,score,aisle
0,Homestyle Waffles,0.145661,0.699007,frozen breakfast
1,Homestyle Belgian Waffles,0.141890,0.680906,frozen breakfast
2,Eggo Buttermilk Waffles,0.138502,0.664652,frozen breakfast
3,Eggo Thick & Fluffy Original Waffles,0.138212,0.663260,frozen breakfast
4,Buttermilk Waffles,0.134741,0.646601,frozen breakfast



Product: Spaghetti  (ID: 32734)


,sub_name,transferability_pct,score,aisle
0,Spaghetti Pasta,0.193824,0.789977,dry pasta
1,Thin Spaghetti Pasta,0.172642,0.703646,dry pasta
2,Whole Grain Spaghetti,0.161338,0.657572,dry pasta
3,Whole Wheat Spaghetti,0.155815,0.635062,dry pasta
4,Penne Rigate,0.106358,0.433486,dry pasta



Product: Diet Coke  (ID: 43631)


,sub_name,transferability_pct,score,aisle
0,Diet Cola,0.184405,0.680563,soft drinks
1,Diet Pepsi Soda,0.152335,0.562203,soft drinks
2,Soda,0.118957,0.439019,soft drinks
3,Fridge Pack Cola,0.112507,0.415215,soft drinks
4,Ginger Ale,0.112360,0.414673,soft drinks


In [23]:
%run ../utils/complements.py

sample_df = pd.read_csv("../results3/sampled-products.csv")
pairwise_df = pd.read_csv("../results3/products-pairwise.csv")

num_orders = order_product_df['order_id'].nunique()
min_co_occurrences = 5  # at least 5 orders
min_pij = min_co_occurrences / num_orders

lift_df = compute_lift(sample_df['product_id'].to_list(), pairwise_df,min_pij=min_pij, total_orders=num_orders)
lift_df.to_csv("../results3/lift.csv", index=False)

complements_df = compute_hybrid_score(lift_df, sample_df['product_id'].to_list(), top_n=10)
complements_df.to_csv("../results3/complements.csv", index=False)

Computing lift: 100%|██████████| 6/6 [00:01<00:00,  4.40it/s]


In [30]:
%run ../utils/complements.py

cii_df = compute_complement_impact_index(complements_df, pairwise_df)
cii_df.to_csv("../results3/complements-cii.csv", index=False)

In [ ]:
%run ../utils/results.py

show_comp_results(complements_df)

Index(['product_id', 'complement_id', 'lift', 'b_complementarity',
       'hybrid_score', 'product_name', 'comp_name', 'aisle_id', 'aisle'],
      dtype='object')

Product: Coke Classic  (ID: 16696)


,comp_name,hybrid_score,aisle
0,Lemon Lime Soda Caffeine Free,0.820062,soft drinks
1,Classic Caffeine Free Soda,0.688852,soft drinks
2,Vanilla Coke,0.550834,soft drinks
3,Ginger Soda,0.491417,soft drinks
4,Chocolate Favorites Fun Size Variety Pack,0.437364,candy chocolate
5,Seasoned Black Cherry Barbecue Pork jerky,0.416736,popcorn jerky
6,Multi-Grain English Muffins,0.373669,breakfast bakery
7,Deluxe Mixed Nuts,0.347049,nuts seeds dried fruit
8,Roasted Garlic Hummus with Pretzels,0.333987,fresh dips tapenades
9,Miniatures Assortment Party Bag,0.322294,candy chocolate



Product: Plain Bagels  (ID: 20738)


,comp_name,hybrid_score,aisle
0,Sliced Pepper Jack Cheese,0.559833,packaged cheese
1,Hearty & Delicious 100% Whole Wheat Bread,0.502512,bread
2,Soft Cream Cheese,0.428556,other creams cheeses
3,Wheat Sandwich Bread,0.375519,bread
4,Double Chocolate Muffins,0.343234,breakfast bakery
5,Clean Burst Liquid Laundry Detergent,0.338362,laundry
6,Extra Sharp White Cheddar Sticks,0.327945,packaged cheese
7,Deluxe Bagels Onion,0.269454,breakfast bakery
8,Regular Cream Cheese Spread,0.259858,other creams cheeses
9,Halloumi Cheese,0.258872,specialty cheeses



Product: Banana  (ID: 24852)


...

,comp_name,hybrid_score,aisle
0,Organic Superfoods Carrot Rice Cakes,0.068897,baby food formula
1,Toddler Cheddar & Leeks Multigrain Wheels Orga...,0.068363,baby food formula
2,Crunchin' 123 Sesame Street Veggie Crackers,0.068086,baby food formula
3,"Crunchin' Grahams, Honey Sticks, 123 Sesame St...",0.058939,baby food formula
4,Organic Pineapple Orange Banana Fruit Yogurt S...,0.055348,baby food formula
5,"Organic Pears, Apples, Peaches, Pumpkin + Cinn...",0.054145,baby food formula
6,Organic Amaze Mint Baby Food,0.053530,baby food formula
7,Smoothie Fruits Squished The Purple One Over 6...,0.052175,baby food formula
8,"Smoothie Fruits, Squished, The Green One, Over...",0.051877,baby food formula
9,"Fruit Snack, 100% Pure, Organic, Peach and Apple",0.050774,baby food formula



Product: Spaghetti  (ID: 32734)


,comp_name,hybrid_score,aisle
0,Shells & White Cheddar,0.172384,instant foods
1,Parmesan Beef Meatballs,0.153591,meat counter
2,Homestyle Tart Cherry Lemonade Drink,0.134526,juice nectars
3,Rao's Homemade Roasted Garlic Sauce,0.120286,pasta sauce
4,Ground Sausage Style Veggie Protein,0.118726,tofu meat alternatives
5,Roasted Garlic Alfredo Pasta Sauce,0.118265,pasta sauce
6,Old World Style Organic Traditional Pasta Sauce,0.117315,pasta sauce
7,Fresh Tilapia Fillets,0.117026,seafood counter
8,Tomato and Basil Bombolina Pasta Sauce,0.112244,pasta sauce
9,Chunky Tomato Garlic & Onion Pasta Sauce,0.111572,pasta sauce



Product: Diet Coke  (ID: 43631)


,comp_name,hybrid_score,aisle
0,Pepsi,1.000000,soft drinks
1,Soft And Strong Double Roll Bath Tissue,0.710828,paper goods
2,Peach Citrus Soda,0.605229,soft drinks
3,Nuggets Assortment,0.564574,candy chocolate
4,Paper Towels White w/ Thirst Pockets,0.519153,paper goods
5,Light Boston Cream Pie Yogurt,0.511912,yogurt
6,Protein Chewy Bar Peanut Butter Dark Chocolate...,0.492086,fruit vegetable snacks
7,Miniatures Assortment Party Bag,0.480176,candy chocolate
8,Juice Drink Variety Pack,0.474706,juice nectars
9,Caffeine Free Diet Coke,0.464558,soft drinks


In [ ]:
%run ../utils/complements.py

pairwise_df = pd.read_csv("../results3/products-pairwise.csv")
complements_df = pd.read_csv("../results3/complements-cii.csv")
network_df = compute_network_enhanced_impact(complements_df, pairwise_df)
network_df.to_csv("../results3/complements-cii-enhanced.csv", index=False)

In [ ]:
product_info_df = pd.read_csv('../data/cleaned/product-features.csv')

totals, details = compute_total_impact(
    pairwise_impact_df=network_df,        # or original_df depending on which impact you want to aggregate
    penetration_df=product_info_df, # or series
    method="topk_weighted",
    top_k=5,
    normalize=True,
    return_details=True
)
totals.to_csv("../results3/complements-total-impact.csv", index=False)

# inspect top total impacts
print(totals.sort_values("total_impact", ascending=False).head(20))


   product_id  total_impact  total_weight  n_complements  mean_pair_impact  \
0       16696      0.000546      0.000801              5          0.706172   
5       43631      0.000448      0.000883              5          0.574459   
1       20738      0.000316      0.000873              5          0.363194   
3       27845      0.000113      0.000585              5          0.190997   
2       24852      0.000101      0.000515              5          0.196059   
4       32734      0.000060      0.000878              5          0.073045   

   total_impact_norm  
0           1.000000  
5           0.820555  
1           0.578623  
3           0.206769  
2           0.184757  
4           0.109187  
